# Breast Cancer Detection
Classify breast tumors as malignant or benign from cell nucleus measurements.

## Project Overview
Binary classification on the sklearn Breast Cancer Wisconsin dataset. Predict malignancy from 30 cell nucleus features.

## Learning Objectives
- Work with medical diagnostic data
- Understand feature importance in cancer detection
- Evaluate classifiers prioritizing recall (minimize missed cancers)

## Problem Statement
Given 30 cell nucleus measurements from a breast mass biopsy, classify the tumor as malignant (0) or benign (1).

## Why This Project Matters
Early cancer detection saves lives. ML models can assist pathologists in screening, reducing missed diagnoses and unnecessary biopsies.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | sklearn.datasets.load_breast_cancer |
| **Target** | target (0=malignant, 1=benign) |
| **Rows** | 569 |
| **Features** | 30 cell nucleus measurements |

## Dataset Source & License
UCI ML Repository via sklearn. Wisconsin Breast Cancer dataset. Public domain.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
TARGET = 'target'

## Dataset Loading

In [4]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame
df['target'] = data.target
print(f'Shape: {df.shape}')
print(f'Class names: {data.target_names}')
df.head()

Shape: (569, 31)
Class names: ['malignant' 'benign']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nTarget distribution:')
print(df[TARGET].value_counts())
print(f'\n0 = malignant, 1 = benign')

Missing: 0
Duplicates: 0

Target distribution:
target
1    357
0    212
Name: count, dtype: int64

0 = malignant, 1 = benign


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#e74c3c','#2ecc71'], edgecolor='black')
axes[0,0].set_title('Malignant vs Benign')

for i, col in enumerate(['mean radius', 'mean texture', 'mean perimeter']):
    if col in df.columns:
        ax = axes[(i+1)//2, (i+1)%2]
        for label in [0, 1]:
            subset = df[df[TARGET] == label]
            ax.hist(subset[col], bins=25, alpha=0.5, label='Malignant' if label==0 else 'Benign')
        ax.set_title(col)
        ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].value_counts())
print(f'\nMalignant rate: {(df[TARGET]==0).mean():.2%}')

target
1    357
0    212
Name: count, dtype: int64

Malignant rate: 37.26%


## Train/Test Split

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (455, 30), Test: (114, 30)


## Preprocessing
All features are numeric. No missing values. Dataset is clean and well-structured.

## Feature Engineering
Using raw cell nucleus features (mean, SE, worst for 10 measurements = 30 features).

## Baseline Model

In [9]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}  Recall={recall_score(y_test, bl_pred):.4f}')

Baseline LR: Acc=0.9649  F1=0.9726  Recall=0.9861


## LazyPredict Benchmark

In [10]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  \
Model                                                                          
PassiveAggressiveClassifier  0.991228           0.988095  0.994709  0.991205   
LogisticRegression           0.982456           0.981151  0.995370  0.982456   
SVC                          0.982456           0.981151  0.995040  0.982456   
SGDClassifier                0.964912           0.967262  0.994378  0.965073   
CalibratedClassifierCV       0.973684           0.964286  0.993717  0.973465   
LinearSVC                    0.964912           0.962302  0.992725  0.964912   
Perceptron                   0.956140           0.960317  0.994048  0.956430   
ExtraTreesClassifier         0.956140           0.950397  0.992725  0.956027   
RandomForestClassifier       0.956140           0.950397  0.993882  0.956027   
KNeighborsClassifier         0.956140           0.950397  0.978836  0.956027   
AdaBoostClassifier           0.956140   

## FLAML AutoML

In [11]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Acc={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb_m = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_m.fit(X_train, y_train)
models['XGBoost'] = xgb_m

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

CatBoost: Acc=0.9561  F1=0.9558
LightGBM: Acc=0.9561  F1=0.9558
XGBoost: Acc=0.9474  F1=0.9471


## Top 3 Model Selection

In [13]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

          Accuracy        F1  Precision    Recall
CatBoost  0.956140  0.955776   0.956869  0.956140
LightGBM  0.956140  0.955776   0.956869  0.956140
XGBoost   0.947368  0.947087   0.947440  0.947368

Top 3: ['CatBoost', 'LightGBM', 'XGBoost']


## Final Evaluation of Top 3

In [14]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

CatBoost: Acc=0.9561  F1=0.9558
              precision    recall  f1-score   support

           0       0.97      0.90      0.94        42
           1       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

LightGBM: Acc=0.9561  F1=0.9558
              precision    recall  f1-score   support

           0       0.97      0.90      0.94        42
           1       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

XGBoost: Acc=0.9474  F1=0.9471
              precision    recall  f1-score   support

           0       0.95      0.90      0.93        42
           1       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94

## Error Analysis

In [15]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

err = (preds != y_test.values).sum()
print(f'Errors: {err}, Error rate: {err/len(y_test):.4f}')

Errors: 5, Error rate: 0.0439


## Interpretation & Business Insight
Worst perimeter, worst concave points, and mean concavity are the strongest cancer indicators. These features capture tumor irregularity.

## Limitations
- Small dataset (569 samples)
- Clean lab data — real clinical data is messier
- Binary classification only (no staging)

## How to Improve
- Add patient demographics
- Multi-class (cancer staging)
- Deep learning on raw histology images

## Production Considerations
- Must prioritize recall — missed cancers are costly
- Regulatory approval (FDA/CE marking)
- Explainable predictions for clinicians

## Common Mistakes
- Optimizing accuracy instead of recall
- Not considering clinical cost of false negatives vs false positives
- Overfitting on small dataset

## Mini Challenge
1. Flip the problem: predict malignant (class 0) as positive
2. Try feature selection with top 10 features
3. Plot ROC curves for all models

## Final Summary
Classified breast tumors from cell measurements. High accuracy achievable (~95%+). Recall for malignant class is the critical metric.

In [16]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "CatBoost": {
    "Accuracy": 0.9561,
    "F1": 0.9558,
    "Precision": 0.9569,
    "Recall": 0.9561
  },
  "LightGBM": {
    "Accuracy": 0.9561,
    "F1": 0.9558,
    "Precision": 0.9569,
    "Recall": 0.9561
  },
  "XGBoost": {
    "Accuracy": 0.9474,
    "F1": 0.9471,
    "Precision": 0.9474,
    "Recall": 0.9474
  },
  "best_model": "CatBoost"
}
